# Wave-window analysis notebook

This notebook is self-contained and repository-friendly. Sensor only needs this notebook and the data folder placed next to it.

It only produces:

- a compact table with the wave parameters
- one common plot for the five selected wave cases

Set CODE_DIR in the next cell if you want to run the notebook from another folder.


In [1]:
from pathlib import Path
import csv
import math
import re

from dataclasses import dataclass

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Change CODE_DIR if your notebook and data folder are stored elsewhere.
CODE_DIR = Path.cwd().resolve()
# Example: CODE_DIR = Path(r"C:/path/to/03_wave_window_analysis")

import math
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np


ANALYSIS_MARGIN_SECONDS = 0.5
MIN_ANALYSIS_DURATION_SECONDS = 3.0
SMOOTHING_WINDOW_SAMPLES = 9

NULL_VALUE = 42.08
MAX_VALUE = 100.0
MAX_CM = 6.0
NULL_CORRECTION_CM = 0.0
CM_PER_VALUE = MAX_CM / (MAX_VALUE - NULL_VALUE)

WATER_DEPTH_M = 0.30
GRAVITY_M_S2 = 9.81

KNOWN_WAVE_STRENGTHS = (60, 66, 70, 76, 80)


@dataclass
class SignalRecord:
    path: Path
    label: str
    wave_strength: int
    distance: int | None
    times: np.ndarray
    raw_values: np.ndarray
    values_cm: np.ndarray


@dataclass(frozen=True)
class AnalysisWindow:
    start_s: float
    end_s: float


def normalize_stem(stem: str) -> str:
    return re.sub(r" \(\d+\)$", "", stem.strip().lower())


def parse_signal_name(path: Path) -> tuple[int, int | None]:
    stem = normalize_stem(path.stem)
    match = re.fullmatch(r"w(\d+)", stem)
    if not match:
        raise ValueError(f"Unsupported filename format: {path.name}")

    digits = match.group(1)
    for wave_strength in KNOWN_WAVE_STRENGTHS:
        prefix = str(wave_strength)
        if digits == prefix:
            return wave_strength, None
        if digits.startswith(prefix):
            distance_text = digits[len(prefix):]
            if distance_text:
                return wave_strength, int(distance_text)

    raise ValueError(f"Could not parse wave strength and distance from {path.name}")


def read_signal_file(path: Path) -> tuple[np.ndarray, np.ndarray]:
    times: list[float] = []
    values: list[float] = []

    with path.open(encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) < 2:
                continue

            try:
                time_value = float(parts[0].replace(",", "."))
                signal_value = float(parts[1].replace(",", "."))
            except ValueError:
                continue

            times.append(time_value)
            values.append(signal_value)

    if not times:
        raise ValueError(f"No numeric signal rows found in {path}")

    return np.array(times, dtype=float), np.array(values, dtype=float)


def convert_to_cm(raw_values: np.ndarray) -> np.ndarray:
    return (raw_values - NULL_VALUE) * CM_PER_VALUE + NULL_CORRECTION_CM


def collect_signal_records(data_dir: Path) -> list[SignalRecord]:
    chosen_files: dict[str, Path] = {}

    for path in sorted(Path(data_dir).glob("w*.txt")):
        normalized = normalize_stem(path.stem)
        existing = chosen_files.get(normalized)
        if existing is None:
            chosen_files[normalized] = path
            continue
        if "(2)" in existing.stem and "(2)" not in path.stem:
            chosen_files[normalized] = path

    records: list[SignalRecord] = []
    for normalized, path in sorted(chosen_files.items()):
        wave_strength, distance = parse_signal_name(path)
        times, raw_values = read_signal_file(path)
        records.append(
            SignalRecord(
                path=path,
                label=normalized.upper(),
                wave_strength=wave_strength,
                distance=distance,
                times=times,
                raw_values=raw_values,
                values_cm=convert_to_cm(raw_values),
            )
        )
    return records


def moving_average(values: np.ndarray, window_samples: int = SMOOTHING_WINDOW_SAMPLES) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or window_samples <= 1:
        return values.copy()

    window_samples = min(window_samples, len(values))
    if window_samples % 2 == 0:
        window_samples -= 1
    if window_samples <= 1:
        return values.copy()

    kernel = np.ones(window_samples, dtype=float) / window_samples
    return np.convolve(values, kernel, mode="same")


def resolve_analysis_window(
    record: SignalRecord,
    analysis_window: AnalysisWindow | tuple[float, float] | None = None,
) -> AnalysisWindow:
    if analysis_window is None:
        start_time = float(record.times[0]) + ANALYSIS_MARGIN_SECONDS
        end_time = float(record.times[-1]) - ANALYSIS_MARGIN_SECONDS
        if end_time - start_time < MIN_ANALYSIS_DURATION_SECONDS:
            start_time = float(record.times[0])
            end_time = float(record.times[-1])
    else:
        if isinstance(analysis_window, AnalysisWindow):
            start_time = float(analysis_window.start_s)
            end_time = float(analysis_window.end_s)
        else:
            start_time = float(analysis_window[0])
            end_time = float(analysis_window[1])

    min_time = float(record.times[0])
    max_time = float(record.times[-1])
    if start_time < min_time or end_time > max_time:
        raise ValueError(
            f"Analysis window for {record.label} must be within {min_time:.3f}-{max_time:.3f} s, "
            f"got {start_time:.3f}-{end_time:.3f} s"
        )
    if end_time <= start_time:
        raise ValueError(f"Analysis window for {record.label} must have end > start")
    if end_time - start_time < float(np.mean(np.diff(record.times))) * 2:
        raise ValueError(f"Analysis window for {record.label} is too short to analyze")

    return AnalysisWindow(start_s=start_time, end_s=end_time)


def extract_analysis_window(
    record: SignalRecord,
    analysis_window: AnalysisWindow | tuple[float, float] | None = None,
    center_on_mean: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    resolved_window = resolve_analysis_window(record, analysis_window)
    mask = (record.times >= resolved_window.start_s) & (record.times <= resolved_window.end_s)
    times_window = record.times[mask]
    values_window_cm = record.values_cm[mask]

    if len(times_window) < 3:
        raise ValueError(f"Too few samples in the analysis window for {record.label}")

    if center_on_mean:
        values_window_cm = values_window_cm - float(np.mean(values_window_cm))

    return times_window, values_window_cm


def estimate_fft_frequency(times_window: np.ndarray, values_window_cm: np.ndarray) -> float:
    if len(times_window) < 3:
        return math.nan

    dt = float(np.mean(np.diff(times_window)))
    if dt <= 0:
        return math.nan

    centered = values_window_cm - float(np.mean(values_window_cm))
    spectrum = np.abs(np.fft.rfft(centered))
    frequencies = np.fft.rfftfreq(len(centered), d=dt)
    if len(spectrum) <= 1:
        return math.nan
    spectrum[0] = 0.0
    return float(frequencies[int(np.argmax(spectrum))])


def estimate_peak_frequency(times_window: np.ndarray, values_window_cm: np.ndarray) -> float:
    centered = values_window_cm - float(np.mean(values_window_cm))
    dt = float(np.mean(np.diff(times_window)))
    if dt <= 0:
        return math.nan

    min_peak_spacing = max(1, int(round(0.35 / dt)))
    peak_times: list[float] = []
    last_peak_index = -min_peak_spacing

    for index in range(1, len(centered) - 1):
        is_local_peak = centered[index] > centered[index - 1] and centered[index] >= centered[index + 1]
        if not is_local_peak or centered[index] <= 0:
            continue
        if index - last_peak_index < min_peak_spacing:
            continue
        peak_times.append(float(times_window[index]))
        last_peak_index = index

    if len(peak_times) < 2:
        return math.nan

    periods = np.diff(np.array(peak_times, dtype=float))
    valid_periods = periods[periods > 0]
    if len(valid_periods) == 0:
        return math.nan

    return float(1.0 / np.mean(valid_periods))


def interpolate_zero_crossing_time(times: np.ndarray, values: np.ndarray, left_index: int, right_index: int) -> float:
    t0 = float(times[left_index])
    t1 = float(times[right_index])
    y0 = float(values[left_index])
    y1 = float(values[right_index])
    if y0 == 0.0:
        return t0
    if y1 == 0.0:
        return t1
    if y1 == y0:
        return 0.5 * (t0 + t1)
    fraction = -y0 / (y1 - y0)
    return float(t0 + fraction * (t1 - t0))


def interpolate_level_crossing_time(
    times: np.ndarray,
    values: np.ndarray,
    left_index: int,
    right_index: int,
    level: float,
) -> float:
    t0 = float(times[left_index])
    t1 = float(times[right_index])
    y0 = float(values[left_index])
    y1 = float(values[right_index])
    if y0 == level:
        return t0
    if y1 == level:
        return t1
    if y1 == y0:
        return 0.5 * (t0 + t1)
    fraction = (level - y0) / (y1 - y0)
    return float(t0 + fraction * (t1 - t0))


def find_zero_crossings(values: np.ndarray, times: np.ndarray) -> tuple[list[tuple[float, int, int]], list[tuple[float, int, int]]]:
    upward: list[tuple[float, int, int]] = []
    downward: list[tuple[float, int, int]] = []
    for index in range(len(values) - 1):
        y0 = float(values[index])
        y1 = float(values[index + 1])
        if y0 <= 0.0 < y1 or y0 < 0.0 <= y1:
            upward.append((interpolate_zero_crossing_time(times, values, index, index + 1), index, index + 1))
        elif y0 >= 0.0 > y1 or y0 > 0.0 >= y1:
            downward.append((interpolate_zero_crossing_time(times, values, index, index + 1), index, index + 1))
    return upward, downward


def find_zero_crossing_before(times: np.ndarray, values: np.ndarray, start_index: int, stop_index: int = 0) -> float:
    for index in range(start_index, stop_index, -1):
        y0 = float(values[index - 1])
        y1 = float(values[index])
        if y0 == 0.0:
            return float(times[index - 1])
        if y1 == 0.0:
            return float(times[index])
        if y0 < 0.0 <= y1 or y0 > 0.0 >= y1:
            return interpolate_zero_crossing_time(times, values, index - 1, index)
    return math.nan


def find_zero_crossing_after(times: np.ndarray, values: np.ndarray, start_index: int, stop_index: int | None = None) -> float:
    upper = len(values) - 1 if stop_index is None else stop_index
    for index in range(start_index, upper):
        y0 = float(values[index])
        y1 = float(values[index + 1])
        if y0 == 0.0:
            return float(times[index])
        if y1 == 0.0:
            return float(times[index + 1])
        if y0 < 0.0 <= y1 or y0 > 0.0 >= y1:
            return interpolate_zero_crossing_time(times, values, index, index + 1)
    return math.nan


def find_level_crossing_before(
    times: np.ndarray,
    values: np.ndarray,
    start_index: int,
    stop_index: int,
    level: float,
) -> float:
    for index in range(start_index, stop_index, -1):
        y0 = float(values[index - 1])
        y1 = float(values[index])
        if (y0 - level) == 0.0:
            return float(times[index - 1])
        if (y1 - level) == 0.0:
            return float(times[index])
        if (y0 - level) * (y1 - level) < 0.0:
            return interpolate_level_crossing_time(times, values, index - 1, index, level)
    return math.nan


def find_level_crossing_after(
    times: np.ndarray,
    values: np.ndarray,
    start_index: int,
    stop_index: int,
    level: float,
) -> float:
    for index in range(start_index, stop_index):
        y0 = float(values[index])
        y1 = float(values[index + 1])
        if (y0 - level) == 0.0:
            return float(times[index])
        if (y1 - level) == 0.0:
            return float(times[index + 1])
        if (y0 - level) * (y1 - level) < 0.0:
            return interpolate_level_crossing_time(times, values, index, index + 1, level)
    return math.nan


def compute_wave_shape_metrics(times_window: np.ndarray, values_window_cm: np.ndarray, frequency_hz: float) -> dict[str, float]:
    geometry_values = np.asarray(values_window_cm, dtype=float)
    centered = geometry_values - float(np.mean(geometry_values))
    std_cm = float(np.std(centered))
    skewness = float(np.mean(centered**3) / (std_cm**3)) if std_cm > 0 else math.nan

    if len(times_window) < 3:
        return _empty_shape_metrics(skewness)

    dt = float(np.mean(np.diff(times_window)))
    if not math.isfinite(dt) or dt <= 0:
        return _empty_shape_metrics(skewness)

    smooth_geometry = moving_average(geometry_values, SMOOTHING_WINDOW_SAMPLES)
    upward_crossings, downward_crossings = find_zero_crossings(smooth_geometry, times_window)

    crest_values_cm: list[float] = []
    trough_values_cm: list[float] = []
    crest_half_height_widths_s: list[float] = []
    trough_half_height_widths_s: list[float] = []
    peak_back_widths_s: list[float] = []
    peak_front_widths_s: list[float] = []
    rise_times_s: list[float] = []
    fall_times_s: list[float] = []
    previous_trough_time_s = math.nan

    for wave_index in range(len(upward_crossings) - 1):
        front_time_s, front_left_index, front_right_index = upward_crossings[wave_index]
        next_front_time_s, _, next_front_right_index = upward_crossings[wave_index + 1]
        rear_candidates = [item for item in downward_crossings if front_time_s < item[0] < next_front_time_s]
        if not rear_candidates:
            continue
        rear_time_s, rear_left_index, rear_right_index = rear_candidates[0]

        crest_slice_start = front_right_index
        crest_slice_end = rear_left_index + 1
        if crest_slice_end <= crest_slice_start:
            continue
        crest_index = crest_slice_start + int(np.argmax(geometry_values[crest_slice_start:crest_slice_end]))
        crest_time_s = float(times_window[crest_index])
        crest_value_cm = float(geometry_values[crest_index])

        trough_slice_start = rear_right_index
        trough_slice_end = next_front_right_index + 1
        if trough_slice_end <= trough_slice_start:
            continue
        trough_index = trough_slice_start + int(np.argmin(geometry_values[trough_slice_start:trough_slice_end]))
        trough_time_s = float(times_window[trough_index])
        trough_value_signed_cm = float(geometry_values[trough_index])
        trough_value_cm = abs(trough_value_signed_cm)

        if not (front_time_s < crest_time_s < rear_time_s < trough_time_s < next_front_time_s):
            continue
        if crest_value_cm <= 0 or trough_value_signed_cm >= 0:
            continue

        peak_back_width_s = float(crest_time_s - front_time_s)
        peak_front_width_s = float(rear_time_s - crest_time_s)
        if peak_back_width_s <= 0 or peak_front_width_s <= 0:
            continue

        crest_half_level_cm = 0.5 * crest_value_cm
        trough_half_level_cm = 0.5 * trough_value_signed_cm
        crest_left_half_time_s = find_level_crossing_before(times_window, geometry_values, crest_index, front_left_index, crest_half_level_cm)
        crest_right_half_time_s = find_level_crossing_after(times_window, geometry_values, crest_index, rear_right_index, crest_half_level_cm)
        trough_left_half_time_s = find_level_crossing_before(times_window, geometry_values, trough_index, rear_left_index, trough_half_level_cm)
        trough_right_half_time_s = find_level_crossing_after(times_window, geometry_values, trough_index, next_front_right_index, trough_half_level_cm)

        if not all(math.isfinite(value) for value in [crest_left_half_time_s, crest_right_half_time_s, trough_left_half_time_s, trough_right_half_time_s]):
            continue

        crest_half_height_width_s = float(crest_right_half_time_s - crest_left_half_time_s)
        trough_half_height_width_s = float(trough_right_half_time_s - trough_left_half_time_s)
        if crest_half_height_width_s <= 0 or trough_half_height_width_s <= 0:
            continue

        crest_values_cm.append(crest_value_cm)
        trough_values_cm.append(trough_value_cm)
        crest_half_height_widths_s.append(crest_half_height_width_s)
        trough_half_height_widths_s.append(trough_half_height_width_s)
        peak_back_widths_s.append(peak_back_width_s)
        peak_front_widths_s.append(peak_front_width_s)

        if math.isfinite(previous_trough_time_s) and previous_trough_time_s < crest_time_s:
            rise_time_s = float(crest_time_s - previous_trough_time_s)
            fall_time_s = float(trough_time_s - crest_time_s)
            if rise_time_s > 0 and fall_time_s > 0:
                rise_times_s.append(rise_time_s)
                fall_times_s.append(fall_time_s)

        previous_trough_time_s = trough_time_s

    crest_cm = float(np.mean(crest_values_cm)) if crest_values_cm else math.nan
    trough_cm = float(np.mean(trough_values_cm)) if trough_values_cm else math.nan
    vertical_asymmetry_ratio = float(crest_cm / trough_cm) if math.isfinite(crest_cm) and math.isfinite(trough_cm) and trough_cm > 0 else math.nan
    crest_half_height_width_s = float(np.mean(crest_half_height_widths_s)) if crest_half_height_widths_s else math.nan
    trough_half_height_width_s = float(np.mean(trough_half_height_widths_s)) if trough_half_height_widths_s else math.nan
    horizontal_half_height_ratio = (
        float(crest_half_height_width_s / trough_half_height_width_s)
        if math.isfinite(crest_half_height_width_s) and math.isfinite(trough_half_height_width_s) and trough_half_height_width_s > 0
        else math.nan
    )
    peak_back_width_s = float(np.mean(peak_back_widths_s)) if peak_back_widths_s else math.nan
    peak_front_width_s = float(np.mean(peak_front_widths_s)) if peak_front_widths_s else math.nan
    horizontal_peak_width_ratio = (
        float(peak_back_width_s / peak_front_width_s)
        if math.isfinite(peak_back_width_s) and math.isfinite(peak_front_width_s) and peak_front_width_s > 0
        else math.nan
    )
    rise_time_s = float(np.mean(rise_times_s)) if rise_times_s else math.nan
    fall_time_s = float(np.mean(fall_times_s)) if fall_times_s else math.nan
    horizontal_asymmetry_ratio = (
        float(np.mean([fall / rise for rise, fall in zip(rise_times_s, fall_times_s) if rise > 0 and fall > 0]))
        if rise_times_s and fall_times_s
        else math.nan
    )

    return {
        "crest_cm": crest_cm,
        "trough_cm": trough_cm,
        "vertical_asymmetry_ratio": vertical_asymmetry_ratio,
        "skewness": float(skewness),
        "crest_half_height_width_s": float(crest_half_height_width_s),
        "trough_half_height_width_s": float(trough_half_height_width_s),
        "horizontal_half_height_ratio": float(horizontal_half_height_ratio),
        "peak_back_width_s": float(peak_back_width_s),
        "peak_front_width_s": float(peak_front_width_s),
        "horizontal_peak_width_ratio": float(horizontal_peak_width_ratio),
        "rise_time_s": float(rise_time_s),
        "fall_time_s": float(fall_time_s),
        "horizontal_asymmetry_ratio": float(horizontal_asymmetry_ratio),
        "crest_front_width_s": float(peak_back_width_s),
        "crest_rear_width_s": float(peak_front_width_s),
    }


def _empty_shape_metrics(skewness: float) -> dict[str, float]:
    return {
        "crest_cm": math.nan,
        "trough_cm": math.nan,
        "vertical_asymmetry_ratio": math.nan,
        "skewness": float(skewness),
        "crest_half_height_width_s": math.nan,
        "trough_half_height_width_s": math.nan,
        "horizontal_half_height_ratio": math.nan,
        "peak_back_width_s": math.nan,
        "peak_front_width_s": math.nan,
        "horizontal_peak_width_ratio": math.nan,
        "rise_time_s": math.nan,
        "fall_time_s": math.nan,
        "horizontal_asymmetry_ratio": math.nan,
        "crest_front_width_s": math.nan,
        "crest_rear_width_s": math.nan,
    }


def solve_wave_number(frequency_hz: float, depth_m: float = WATER_DEPTH_M) -> float:
    if not math.isfinite(frequency_hz) or frequency_hz <= 0:
        return math.nan

    omega = 2.0 * math.pi * frequency_hz
    k = max((omega**2) / GRAVITY_M_S2, 1e-6)
    for _ in range(100):
        tanh_term = math.tanh(k * depth_m)
        sech_term = 1.0 / math.cosh(k * depth_m)
        function = GRAVITY_M_S2 * k * tanh_term - omega**2
        derivative = GRAVITY_M_S2 * (tanh_term + k * depth_m * sech_term * sech_term)
        if derivative == 0:
            break
        next_k = k - function / derivative
        if next_k <= 0:
            next_k = k / 2.0
        if abs(next_k - k) < 1e-12:
            k = next_k
            break
        k = next_k
    return float(k)


def summarize_record(
    record: SignalRecord,
    analysis_window: AnalysisWindow | tuple[float, float] | None = None,
    center_on_mean: bool = False,
) -> dict[str, object]:
    times_window, values_window_cm = extract_analysis_window(record, analysis_window, center_on_mean=center_on_mean)
    fft_frequency_hz = estimate_fft_frequency(times_window, values_window_cm)
    peak_frequency_hz = estimate_peak_frequency(times_window, values_window_cm)

    if math.isfinite(fft_frequency_hz) and math.isfinite(peak_frequency_hz):
        frequency_hz = float((fft_frequency_hz + peak_frequency_hz) / 2.0)
    elif math.isfinite(fft_frequency_hz):
        frequency_hz = float(fft_frequency_hz)
    else:
        frequency_hz = float(peak_frequency_hz)

    period_s = math.nan if not math.isfinite(frequency_hz) or frequency_hz <= 0 else 1.0 / frequency_hz
    omega_rad_s = math.nan if not math.isfinite(frequency_hz) else 2.0 * math.pi * frequency_hz
    wave_number_m_inv = solve_wave_number(frequency_hz)
    wavelength_m = math.nan if not math.isfinite(wave_number_m_inv) or wave_number_m_inv <= 0 else (2.0 * math.pi / wave_number_m_inv)

    min_cm = float(np.min(values_window_cm))
    max_cm = float(np.max(values_window_cm))
    amplitude_cm = 0.5 * (max_cm - min_cm)
    shape_metrics = compute_wave_shape_metrics(times_window, values_window_cm, frequency_hz)
    wave_height_cm = (
        float(shape_metrics["crest_cm"] + shape_metrics["trough_cm"])
        if math.isfinite(shape_metrics["crest_cm"]) and math.isfinite(shape_metrics["trough_cm"])
        else math.nan
    )
    wave_steepness = (
        float((wave_height_cm / 100.0) / wavelength_m)
        if math.isfinite(wave_height_cm) and math.isfinite(wavelength_m) and wavelength_m > 0
        else math.nan
    )

    return {
        "label": record.label,
        "wave_strength": record.wave_strength,
        "distance": record.distance,
        "analysis_start_s": float(times_window[0]),
        "analysis_end_s": float(times_window[-1]),
        "mean_cm": float(np.mean(values_window_cm)),
        "min_cm": min_cm,
        "max_cm": max_cm,
        "amplitude_cm": float(amplitude_cm),
        "period_s": float(period_s),
        "frequency_hz": float(frequency_hz),
        "fft_frequency_hz": float(fft_frequency_hz),
        "peak_frequency_hz": float(peak_frequency_hz),
        "frequency_delta_hz": float(abs(fft_frequency_hz - peak_frequency_hz))
        if math.isfinite(fft_frequency_hz) and math.isfinite(peak_frequency_hz)
        else math.nan,
        "omega_rad_s": float(omega_rad_s),
        "wave_number_m_inv": float(wave_number_m_inv),
        "wavelength_m": float(wavelength_m),
        "crest_cm": float(shape_metrics["crest_cm"]),
        "trough_cm": float(shape_metrics["trough_cm"]),
        "vertical_asymmetry_ratio": float(shape_metrics["vertical_asymmetry_ratio"]),
        "skewness": float(shape_metrics["skewness"]),
        "crest_half_height_width_s": float(shape_metrics["crest_half_height_width_s"]),
        "trough_half_height_width_s": float(shape_metrics["trough_half_height_width_s"]),
        "horizontal_half_height_ratio": float(shape_metrics["horizontal_half_height_ratio"]),
        "peak_back_width_s": float(shape_metrics["peak_back_width_s"]),
        "peak_front_width_s": float(shape_metrics["peak_front_width_s"]),
        "horizontal_peak_width_ratio": float(shape_metrics["horizontal_peak_width_ratio"]),
        "wave_height_cm": float(wave_height_cm),
        "wave_steepness": float(wave_steepness),
        "rise_time_s": float(shape_metrics["rise_time_s"]),
        "fall_time_s": float(shape_metrics["fall_time_s"]),
        "horizontal_asymmetry_ratio": float(shape_metrics["horizontal_asymmetry_ratio"]),
        "crest_front_width_s": float(shape_metrics["crest_front_width_s"]),
        "crest_rear_width_s": float(shape_metrics["crest_rear_width_s"]),
    }


def format_distance(distance: int | None) -> str:
    return "pure" if distance is None else str(distance)

DATA_DIR = CODE_DIR / "sensor_data_wave_zoom" / "wave_data"
RESULT_DIR = CODE_DIR / "sensor_data_wave_zoom" / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_WINDOWS = {
    "W60": AnalysisWindow(6.0, 10.0),
    "W66": AnalysisWindow(6.0, 10.0),
    "W70": AnalysisWindow(6.0, 10.0),
    "W76": AnalysisWindow(6.0, 10.0),
    "W80": AnalysisWindow(6.0, 10.0),
}

records = sorted(
    collect_signal_records(DATA_DIR),
    key=lambda record: record.wave_strength,
)

print(f"Notebook directory: {CODE_DIR}")
print(f"Data folder:        {DATA_DIR}")
print(f"Results folder:     {RESULT_DIR}")
print("Loaded signals:")
for record in records:
    window = ANALYSIS_WINDOWS.get(record.label)
    print(f" - {record.label}: time = {record.times[0]:.3f} to {record.times[-1]:.3f} s | window = {window.start_s:.3f} to {window.end_s:.3f} s")


Notebook folder: C:\Users\knutm\Documents\Skole\Master\Python\bÃ¸lge
Data folder:     C:\Users\knutm\Documents\Skole\Master\Python\bÃ¸lge\sensor_data_wave_zoom\wave_data
Results folder:  C:\Users\knutm\Documents\Skole\Master\Python\bÃ¸lge\sensor_data_wave_zoom\results
Loaded signals:
 - W60: time = 0.000 to 12.830 s | window = 6.000 to 10.000 s
 - W66: time = 0.000 to 13.080 s | window = 6.000 to 10.000 s
 - W70: time = 0.000 to 10.820 s | window = 6.000 to 10.000 s
 - W76: time = 0.000 to 12.980 s | window = 6.000 to 10.000 s
 - W80: time = 0.000 to 12.720 s | window = 6.000 to 10.000 s


In [2]:
# This cell builds the compact wave table and saves it as a CSV file.

summary_rows = []

def local_smooth_signal(values, window_size=9):
    values = np.asarray(values, dtype=float)
    if window_size <= 1:
        return values.copy()
    kernel = np.ones(window_size, dtype=float) / window_size
    return np.convolve(values, kernel, mode="same")

def interpolate_zero_crossing_time(times, values, left_index, right_index):
    t0 = float(times[left_index])
    t1 = float(times[right_index])
    y0 = float(values[left_index])
    y1 = float(values[right_index])
    if y0 == 0.0:
        return t0
    if y1 == 0.0:
        return t1
    if y1 == y0:
        return 0.5 * (t0 + t1)
    fraction = -y0 / (y1 - y0)
    return float(t0 + fraction * (t1 - t0))

def find_zero_crossings(times, values):
    upward = []
    downward = []
    for index in range(len(values) - 1):
        y0 = float(values[index])
        y1 = float(values[index + 1])
        if y0 <= 0.0 < y1 or y0 < 0.0 <= y1:
            upward.append((interpolate_zero_crossing_time(times, values, index, index + 1), index, index + 1))
        elif y0 >= 0.0 > y1 or y0 > 0.0 >= y1:
            downward.append((interpolate_zero_crossing_time(times, values, index, index + 1), index, index + 1))
    return upward, downward

for record in records:
    window = ANALYSIS_WINDOWS.get(record.label)
    row = summarize_record(record, analysis_window=window)

    times_window, values_window_cm = extract_analysis_window(record, analysis_window=window)
    centered_window_cm = np.asarray(values_window_cm, dtype=float) - float(np.mean(values_window_cm))
    relative_times_s = times_window - float(times_window[0])
    smooth_centered_cm = local_smooth_signal(centered_window_cm, window_size=9)
    upward_crossings, downward_crossings = find_zero_crossings(relative_times_s, smooth_centered_cm)

    peak_back_widths_s = []
    peak_front_widths_s = []

    for wave_index in range(len(upward_crossings) - 1):
        back_time_s, back_left_index, back_right_index = upward_crossings[wave_index]
        next_back_time_s, _, _ = upward_crossings[wave_index + 1]
        front_candidates = [item for item in downward_crossings if back_time_s < item[0] < next_back_time_s]
        if not front_candidates:
            continue
        front_time_s, front_left_index, _ = front_candidates[0]
        crest_slice_start = back_right_index
        crest_slice_end = front_left_index + 1
        if crest_slice_end <= crest_slice_start:
            continue
        crest_local_index = int(np.argmax(centered_window_cm[crest_slice_start:crest_slice_end]))
        crest_index = crest_slice_start + crest_local_index
        crest_time_s = float(relative_times_s[crest_index])
        crest_value_cm = float(centered_window_cm[crest_index])
        if not (back_time_s < crest_time_s < front_time_s < next_back_time_s):
            continue
        if crest_value_cm <= 0:
            continue
        wb_s = float(crest_time_s - back_time_s)
        wf_s = float(front_time_s - crest_time_s)
        if wb_s <= 0 or wf_s <= 0:
            continue
        peak_back_widths_s.append(wb_s)
        peak_front_widths_s.append(wf_s)

    R_wb_wf = (
        float(np.mean(peak_back_widths_s) / np.mean(peak_front_widths_s))
        if peak_back_widths_s and peak_front_widths_s and np.mean(peak_front_widths_s) > 0
        else math.nan
    )

    summary_rows.append({
        "Wave case": record.label,
        "a [cm]": row["amplitude_cm"],
        "T [s]": row["period_s"],
        "f [Hz]": row["frequency_hz"],
        "k [1/m]": row["wave_number_m_inv"],
        "lambda [m]": row["wavelength_m"],
        "R_wb/wf [-]": R_wb_wf,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Wave case").reset_index(drop=True)
display(summary_df.round(6))

summary_df.to_csv(RESULT_DIR / "wave_summary_table.csv", index=False, encoding="utf-8-sig")
print(f"Saved table: {RESULT_DIR / 'wave_summary_table.csv'}")


,Wave case,a [cm],T [s],f [Hz],k [1/m],lambda [m],R_wb/wf [-]
0,W60,0.633978,0.503510,1.986058,15.875880,0.395769,0.901770
1,W66,0.715815,0.456270,2.191686,19.331052,0.325031,1.355898
2,W70,0.789365,0.443432,2.255137,20.466365,0.307001,1.062733
3,W76,0.801796,0.404106,2.474597,24.643375,0.254964,1.156295
4,W80,0.903315,0.396204,2.523950,25.636119,0.245091,1.446243


Saved table: C:\Users\knutm\Documents\Skole\Master\Python\bÃ¸lge\sensor_data_wave_zoom\results\wave_summary_table.csv


In [3]:
fig, axes = plt.subplots(len(records), 1, figsize=(14, 3.6 * len(records)), sharex=True)
if len(records) == 1:
    axes = [axes]

for ax, record in zip(axes, records):
    window = ANALYSIS_WINDOWS.get(record.label)
    times_window, values_window_cm = extract_analysis_window(record, analysis_window=window, center_on_mean=False)
    relative_times_s = times_window - float(times_window[0])
    ax.plot(relative_times_s, values_window_cm, linewidth=1.6, label=record.label)
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1.0, alpha=0.6)
    ax.set_title(record.label, fontsize=12, pad=6)
    ax.set_ylabel("Distance [cm]")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Time from window start [s]")
fig.suptitle("Common plot for all five wave cases from the selected windows", fontsize=16, fontweight="bold", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.985))

plot_path = RESULT_DIR / "common_wave_plot.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved plot: {plot_path}")


Saved plot: C:\Users\knutm\Documents\Skole\Master\Python\bÃ¸lge\sensor_data_wave_zoom\results\common_wave_plot.png
